In [ ]:
#| default_exp bench

# bench

> Fit, time, and score single model configurations for benchmarking

This module is the scoring layer behind the README benchmark tables and `tools/accuracy.py`: load a dataset with `fastforest.datasets`, run `fit_scores` over the models you care about, read the comparison, and merge the numbers into `tools/results/*.csv`. Results come back as objects whose bare display is the report.

In [ ]:
#| export
import multiprocessing as mp
import json,os,platform,signal,sqlite3,subprocess,time,traceback
from datetime import datetime
from pathlib import Path

import numpy as np,pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier,HistGradientBoostingRegressor,RandomForestClassifier,RandomForestRegressor
from sklearn.metrics import f1_score,log_loss,mean_squared_error,r2_score
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split

from fastcore.script import is_cli
from fastforest.sklearn import sklearn_hist_preprocessor,sklearn_preprocessor
from fastforest.datasets import Dataset,_rows,dataset_task,load_data,order_column,split_indices


Importing `fastforest.bench` switches pandas to plain-text `DataFrame` reprs, so dataframes display compactly in any session that loads the benchmark machinery.

In [ ]:
#| export
pd.set_option("display.notebook_repr_html", False)

In [ ]:
import tempfile,shutil
import fastforest.bench
from fastcore.test import *
from fastforest.datasets import load_data,split_indices

## Result objects

A `FitResult` is the outcome of scoring one model once. It is a plain dict, so every recorded number stays subscriptable, but its display leads with the headline entries in a fixed order: task metrics, then timings, then forest size. Bulky diagnostics (importance arrays, split counts) are counted rather than shown.

In [ ]:
#| export
_headline = ("rmse", "r2", "f1", "log_loss", "fit", "predict", "n_trees", "status")

def _fmt(value): return f"{value:#.4g}" if isinstance(value, float) else str(value)

class FitResult(dict):
    "`fit_score` results: a dict with a headline-first display; diagnostics stay subscriptable."
    def __init__(self, *args, model=None, dataset=None, task=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.model,self.dataset,self.task = model,dataset,task

    @property
    def label(self): return " on ".join(part for part in (self.model, self.dataset) if part)

    def headline(self):
        "The metric, timing, and size entries, in display order."
        return [(key, self[key]) for key in _headline if key in self]

    def split_frame(self, encoded=False):
        "Return split-gain and depth diagnostics by logical or encoded feature."
        prefix = "encoded_" if encoded else ""
        names,counts = self[f"{prefix}feature_names"],np.asarray(self[f"{prefix}split_diagnostics"])
        result = pd.DataFrame(dict(feature=names, importance=self[f"{prefix}feature_importances"], splits=counts[:,0], equality=counts[:,1],
            depth_0_3=counts[:,2], depth_4_7=counts[:,3], depth_8_plus=counts[:,0]-counts[:,2]-counts[:,3]))
        return result.sort_values("splits", ascending=False, ignore_index=True)


    def __repr__(self):
        pairs = self.headline()
        extra = len(self)-len(pairs)
        tail = f" (+{extra} diagnostics)" if extra else ""
        label = f"{self.label}: " if self.label else ""
        return label + ", ".join(f"{key} {_fmt(value)}" for key,value in pairs) + tail

A literal result shows the display rules without fitting anything. The `model` and `dataset` labels are not dict entries; they ride on the object and form its `label`.

In [ ]:
r = FitResult(dict(rmse=5.844, r2=0.868, fit=0.0124, predict=0.0006, n_trees=64, feature_names=("age", "cement")),
    model="FastForest", dataset="concrete", task="regression")
test_eq(r.label, "FastForest on concrete")
test_eq(r["n_trees"], 64)
r

FastForest on concrete: rmse 5.844, r2 0.8680, fit 0.01240, predict 0.0006000, n_trees 64 (+1 diagnostics)

A `FitResults` collects one `FitResult` per model. It is a list, but indexes by model name, and displays as a comparison table (markdown in notebooks, one line per model in a terminal).

In [ ]:
#| export
class FitResults(list):
    "Several `FitResult`s, displayed as one table and optionally persisted as a resumable run."
    def __getitem__(self, key):
        if not isinstance(key, str): return super().__getitem__(key)
        matches = [result for result in self if result.model == key]
        if len(matches) == 1: return matches[0]
        if matches: raise KeyError(f"several results for model {key!r}; select a dataset first")
        raise KeyError(f"no result for model {key!r}; have {sorted({result.model for result in self})}")

    def __repr__(self): return "\n".join(repr(result) for result in self)

    def _repr_markdown_(self):
        keys = [key for key in _headline if any(key in result for result in self)]
        head = "| result | " + " | ".join(keys) + " |"
        rule = "|---" * (len(keys)+1) + "|"
        rows = ("| " + (result.label or "?") + " | " + " | ".join(_fmt(result[key]) if key in result else "" for key in keys) + " |"
            for result in self)
        return "\n".join((head, rule, *rows))

    def for_dataset(self, dataset):
        "Select one dataset's results."
        return FitResults(result for result in self if result.dataset == dataset or str(result.dataset) == str(dataset))

    def frame(self):
        "Return the compact metrics, timings, and tree summaries as a dataframe."
        rows = []
        for result in self:
            row = dict(dataset=str(result.dataset), task=result.task, model=result.model)
            row.update((key,result[key]) for key in _headline if key in result)
            for key in ("n_logical_features", "n_encoded_features", "n_physical_features", "n_projected_features"):
                if key in result: row[key] = result[key]
            for key in ("tree_nodes", "tree_leaves", "tree_depth"):
                if key in result: row[f"{key}_mean"],row[f"{key}_median"] = result[key]
            rows.append(row)
        return pd.DataFrame(rows)

    def _shared(self, name):
        values = {getattr(result, name) for result in self}
        if len(values) != 1: raise ValueError(f"results span several {name}s: {sorted(map(str, values))}")
        return values.pop()

    def save(self, results_dir):
        "Merge one dataset's results into the README benchmark CSVs."
        results = {result.model:None if result.get("status") == "timeout" else result for result in self}
        return save_results(results, self._shared("task"), self._shared("dataset"), results_dir)

    def save_run(self, path):
        "Upsert these rows into a resumable full-precision comparison CSV."
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        fresh = self.frame()
        if path.exists():
            current = pd.read_csv(path)
            keys = set(zip(fresh.dataset, fresh.model))
            current = current[[key not in keys for key in zip(current.dataset, current.model)]]
            fresh = pd.concat([current,fresh], ignore_index=True)
        fresh.to_csv(path, index=False)
        return path

In [ ]:
rf = FitResult(dict(rmse=5.474, r2=0.884, fit=0.068, predict=0.015), model="RandomForest", dataset="concrete", task="regression")
results = FitResults([r, rf])
results

| model | rmse | r2 | fit | predict | n_trees |
|---|---|---|---|---|---|
| FastForest on concrete | 5.844 | 0.8680 | 0.01240 | 0.0006000 | 64 |
| RandomForest on concrete | 5.474 | 0.8840 | 0.06800 | 0.01500 |  |

In [ ]:
test_eq(results["RandomForest"], rf)
test_fail(lambda: results["HistGBM"], contains="no result for model")
results[0]

FastForest on concrete: rmse 5.844, r2 0.8680, fit 0.01240, predict 0.0006000, n_trees 64 (+1 diagnostics)

## Scoring models

`make_model` builds one ready-to-fit estimator. `FastForest` takes the raw data directly. The sklearn models get the benchmark's reproducible preprocessing pipelines from `fastforest.sklearn` when the data is a dataframe: `RandomForest` with one-hot and target encoding, `HistGBM` with its native categoricals.

In [ ]:
#| export
def make_model(name, task, X_train, missing_values, seed, rf_trees, ff_kwargs):
    if name == "FastForest":
        from fastforest import FastForest,FastForestClassifier
        params = dict(ff_kwargs, seed=seed, missing_values=missing_values)
        params.pop("split_prior_rows" if task == "classification" else "class_weight_power", None)
        return (FastForestClassifier if task == "classification" else FastForest)(**params)
    if name == "RandomForest":
        forest = RandomForestClassifier if task == "classification" else RandomForestRegressor
        model = forest(n_estimators=rf_trees, n_jobs=-1, random_state=seed)
        if not hasattr(X_train, "columns"): return model
        target_type = "auto" if task == "classification" else "continuous"
        preprocessor = sklearn_preprocessor(X_train, missing_values, target_type=target_type)
        return make_pipeline(preprocessor, model)
    hist = HistGradientBoostingClassifier if task == "classification" else HistGradientBoostingRegressor
    if not hasattr(X_train, "columns"): return hist(random_state=seed)
    preprocessor,categories = sklearn_hist_preprocessor(X_train, missing_values, target_type="auto" if task == "classification" else "continuous")
    return make_pipeline(preprocessor, hist(random_state=seed, categorical_features=categories))

`fit_score` fits one named model, timing fit and prediction separately, and scores the validation data: RMSE and R² for regression, macro-F1 and log loss for classification. FastForest results also carry tree-shape and importance diagnostics. `AutoForest` and `Autogrow` rows delegate to `fit_score_auto`, which records the sizing screen's choices as well; sizing only activates above its row thresholds, so on small data those rows report `sizing_active=False`.

In [ ]:
#| export
def fit_score_auto(task, X_train, y_train, X_test, y_test, missing_values, ff_kwargs, autogrow, dataset=None):
    from fastforest.auto import AutoForest,AutoForestClassifier
    start = time.perf_counter()
    params = {name:value for name,value in ff_kwargs.items() if name != "n_trees"}
    params.pop("split_prior_rows" if task == "classification" else "class_weight_power", None)
    model = (AutoForestClassifier if task == "classification" else AutoForest)(seed=42, autogrow=autogrow, max_trees=192,
        missing_values=missing_values, **params)
    model.fit(X_train, y_train)
    result = dict(fit=time.perf_counter()-start, sizing=model.sizing_,
        sizing_active=bool(model.sizing_["changes"]) or (autogrow and model.n_trees_ > model.tree_batch_size),
        selected=f"bootstrap_max={model.bootstrap_max}, max_node_samples={model.max_node_samples}, max_features={model.max_features}, target_statistics={model.target_statistics}, trees={model.n_trees_}",
        selection_method="autogrow" if autogrow else "AutoForest", selection_scores=model.tree_history_,
        n_trees=model.n_trees_, tree_nodes=(float(np.mean(model.tree_node_counts_)),float(np.median(model.tree_node_counts_))),
        tree_leaves=(float(np.mean(model.tree_leaf_counts_)),float(np.median(model.tree_leaf_counts_))),
        tree_depth=(float(np.mean(model.tree_depths_)),float(np.median(model.tree_depths_))))
    start = time.perf_counter()
    if task == "classification":
        probabilities = model.predict_proba(X_test)
        predictions = model.classes_[probabilities.argmax(axis=1)]
        result.update(predict=time.perf_counter()-start, f1=f1_score(y_test, predictions, average="macro"),
            log_loss=log_loss(y_test, probabilities, labels=model.classes_), prediction_trees_per_batch=model.prediction_trees_per_batch_)
    else:
        predictions = model.predict(X_test)
        result.update(predict=time.perf_counter()-start, rmse=mean_squared_error(y_test, predictions)**.5, r2=r2_score(y_test, predictions))
    return FitResult(result, model="Autogrow" if autogrow else "AutoForest", dataset=dataset, task=task)

In [ ]:
#| export
def fit_score(name, task, X_train, y_train, X_test, y_test, missing_values, rf_trees, ff_kwargs, dataset=None):
    "Fit, time, and score one model."
    if name in ("AutoForest","Autogrow"):
        return fit_score_auto(task, X_train, y_train, X_test, y_test, missing_values, ff_kwargs, name=="Autogrow", dataset)
    result = {}
    start = time.perf_counter()
    model = make_model(name, task, X_train, missing_values, 42, rf_trees, ff_kwargs)
    model.fit(X_train, y_train)
    result["fit"] = time.perf_counter()-start
    if name == "FastForest":
        result["n_trees"] = model.n_trees_
        result["tree_nodes"] = float(np.mean(model.tree_node_counts_)),float(np.median(model.tree_node_counts_))
        result["tree_leaves"] = float(np.mean(model.tree_leaf_counts_)),float(np.median(model.tree_leaf_counts_))
        result["tree_depth"] = float(np.mean(model.tree_depths_)),float(np.median(model.tree_depths_))
        result["feature_importances"] = np.asarray(model.feature_importances_)
        result["encoded_feature_importances"] = np.asarray(model._model.feature_importances)
        result["feature_names"] = tuple(map(str, model.feature_names_in_))
        result["encoded_feature_names"] = tuple(model._encoder.encoded_names)
        phys = np.asarray(model._encoder.fit_layout[3])
        direct = phys[phys != np.iinfo(np.uint32).max]
        result.update(n_logical_features=len(model.feature_names_in_), n_encoded_features=len(model._encoder.encoded_names),
            n_physical_features=len(np.unique(direct)), n_projected_features=int((phys == np.iinfo(np.uint32).max).sum()))
        result["bundles"] = [(column.name,column.members) for column in model.column_info_ if column.members]
        result["split_diagnostics"] = np.zeros((len(model.feature_importances_), 4), dtype=np.int64)
        if hasattr(model._model, "split_counts_by_depth"):
            depth_counts = np.asarray(model._model.split_counts_by_depth)
            encoded_diagnostics = np.column_stack((depth_counts[:,:,0].sum(axis=1), depth_counts[:,:,1].sum(axis=1),
                depth_counts[:,:4,0].sum(axis=1), depth_counts[:,4:8,0].sum(axis=1)))
            result["encoded_split_diagnostics"] = encoded_diagnostics
            raw_ids = np.asarray(model._encoder._native.encoded_to_raw)
            np.add.at(result["split_diagnostics"], raw_ids, encoded_diagnostics)
        if task == "classification": result["prediction_trees_per_batch"] = model.prediction_trees_per_batch_
    if name == "HistGBM": result["iterations"] = model.n_iter_ if hasattr(model, "n_iter_") else model[-1].n_iter_
    start = time.perf_counter()
    if task == "classification":
        probabilities = model.predict_proba(X_test)
        predictions = model.classes_[probabilities.argmax(axis=1)]
    else: predictions = model.predict(X_test)
    if "predict" not in result: result["predict"] = time.perf_counter()-start
    if task == "classification":
        result["f1"] = f1_score(y_test, predictions, average="macro")
        result["log_loss"] = log_loss(y_test, probabilities, labels=model.classes_)
    else:
        result["rmse"] = mean_squared_error(y_test, predictions)**0.5
        result["r2"] = r2_score(y_test, predictions)
    return FitResult(result, model=name, dataset=dataset, task=task)

The examples below score the 1,030-row Concrete Compressive Strength dataset on its canonical split (`y=None`: stratification labels are only for classification).

In [ ]:
name,X,y,missing,task = load_data(Dataset.concrete, "../data")
train,valid,split = split_indices(Dataset.concrete, X, None)
one = fit_score("FastForest", task, X[train], y[train], X[valid], y[valid], missing, 100, {}, dataset=Dataset.concrete)
test_eq(one.task, "regression")
assert one["rmse"] < 10
one

FastForest on concrete: rmse 5.698, r2 0.8740, fit 0.005744, predict 0.0005449, n_trees 64 (+7 diagnostics)

`fit_scores` runs several models on the same split and returns the `FitResults`, so the whole comparison is three expressions: load, score, display.

In [ ]:
#| export
def fit_scores(
    models, # Model names to run, e.g. ('FastForest', 'RandomForest', 'HistGBM')
    task, # 'regression' or 'classification'
    X_train, y_train, X_test, y_test, missing_values,
    rf_trees=100, # sklearn RF trees
    ff_kwargs=None, # FastForest keyword overrides
    dataset=None, # Optional `Dataset` member (or its string id), for labels and `FitResults.save`
):
    "Run `fit_score` for each named model, returning `FitResults`."
    return FitResults(fit_score(name, task, X_train, y_train, X_test, y_test, missing_values, rf_trees, ff_kwargs or {}, dataset=dataset)
        for name in models)

In [ ]:
results = fit_scores(("FastForest", "RandomForest", "HistGBM"), task,
    X[train], y[train], X[valid], y[valid], missing, dataset=Dataset.concrete)
results

| model | rmse | r2 | fit | predict | n_trees |
|---|---|---|---|---|---|
| FastForest on concrete | 5.698 | 0.8740 | 0.004437 | 0.0005328 | 64 |
| RandomForest on concrete | 5.474 | 0.8837 | 0.05075 | 0.01260 |  |
| HistGBM on concrete | 4.646 | 0.9162 | 1.180 | 0.006728 |  |

## Isolated runs

`call_timed` runs one importable callable in a fresh `spawn` process and hands back its result. Because the child is a new interpreter, it loads the compiled extension as `maturin develop` last built it, and bench itself defers importing the model classes into `make_model` and `fit_score_auto`: a long-lived session can keep its datasets in memory and measure every rebuild through `call_timed`, with no kernel restarts. Startup is bounded separately (a readiness handshake with a 30-second limit), so `timeout` measures the call itself: on expiry the child and any workers it spawned are killed as a process group and the result is `None`, which is how `tools/accuracy.py` marks a model as timed out. A child error re-raises in the parent, with the child's traceback as the message. Spawn pickles a callable by module path, so `target` must be importable: pass `fastforest.bench.fit_score`, not a function defined in a notebook or kernel session, and `call_timed` resolves its own child shim the same way so that it works wherever it was imported from.

In [ ]:
#| export
def _child_call(send, target, args, kwargs):
    try: os.setsid()
    except (AttributeError, PermissionError): pass
    try:
        send.send(("ready", None))
        send.send((True, target(*args, **kwargs)))
    except BaseException: send.send((False, traceback.format_exc()))
    finally: send.close()

def _stop(process):
    "Stop a child and any workers it created."
    try: os.killpg(process.pid, signal.SIGTERM)
    except (AttributeError, ProcessLookupError): process.terminate()
    process.join(5)
    if process.is_alive():
        try: os.killpg(process.pid, signal.SIGKILL)
        except (AttributeError, ProcessLookupError): process.kill()
        process.join()

def _receive(receive, process):
    try: return receive.recv()
    except EOFError: raise RuntimeError(f"subprocess exited with code {process.exitcode} without a result") from None

def call_timed(
    target, # Importable callable, run in a fresh `spawn` process
    *args, # Positional arguments for `target`
    timeout=None, # Seconds allowed after startup before the child is killed; None waits forever
    **kwargs, # Keyword arguments for `target`
):
    "Run `target(*args, **kwargs)` in a fresh process: its result, or None on timeout."
    from fastforest.bench import _child_call
    context = mp.get_context("spawn")
    receive,send = context.Pipe(False)
    process = context.Process(target=_child_call, args=(send, target, args, kwargs))
    process.start()
    send.close()
    if not receive.poll(30):
        _stop(process)
        raise RuntimeError("subprocess did not start within 30 seconds")
    status,result = _receive(receive, process)
    if status != "ready": raise RuntimeError(result)
    process.join(timeout)
    if process.is_alive():
        _stop(process)
        return None
    succeeded,result = _receive(receive, process)
    if not succeeded: raise RuntimeError(result)
    return result


## Run log

Benchmark history answers before/after questions that the latest row cannot: regressions, machine differences, and the effect of a default change. `bench_db` opens a local sqlite run log, creating it on first use, and `record_result` appends one run with its metrics, timings, configuration, and environment. The log lives at `tools/results/bench.db`, is gitignored, and is machine-local. When that directory is absent, as in an installed package or a working directory outside the repository, recording skips and returns `None`. The benchmark tooling therefore works anywhere, and records when run from the repository root.


In [ ]:
#| export
BENCH_DB = Path("tools/results/bench.db")

_RUNS_SCHEMA = """create table if not exists runs (
    id integer primary key, ts text, machine text, version text, dataset text, model text, task text, split text,
    status text, rmse real, r2 real, f1 real, log_loss real, fit real, predict real, n_trees integer,
    dataset_rows integer, dataset_features integer, config text, detail text)"""

def _machine():
    try: return subprocess.check_output(["sysctl","-n","machdep.cpu.brand_string"], text=True).strip()
    except Exception: return platform.processor() or platform.machine()

def _version():
    from importlib.metadata import version
    return version("fastforest")

def bench_db(path=BENCH_DB):
    "Open the benchmark run-log database, creating it on first use."
    db = sqlite3.connect(path)
    db.row_factory = sqlite3.Row
    db.execute(_RUNS_SCHEMA)
    return db

def record_result(
    result, # `FitResult` to record; gains a `run_id` entry
    config=None, # The call configuration to store alongside the metrics
    db_path=BENCH_DB, # Run-log database; recording skips when its directory is absent
):
    "Append one run to the local run log, returning its row id (None when recording skips)."
    path = Path(db_path)
    if not path.parent.is_dir(): return None
    detail = {key:result[key] for key in ("sizing_active","selected","n_encoded_features") if key in result}
    row = dict(ts=datetime.now().isoformat(timespec="seconds"), machine=_machine(), version=_version(),
        dataset=str(result.dataset), model=result.model, task=result.task, split=result.get("split"),
        status=result.get("status"), n_trees=result.get("n_trees"), dataset_rows=result.get("dataset_rows"),
        dataset_features=result.get("dataset_features"), config=json.dumps(config or {}, default=str), detail=json.dumps(detail))
    for key in ("rmse","r2","f1","log_loss","fit","predict"): row[key] = result.get(key)
    db = bench_db(path)
    with db: result["run_id"] = db.execute(f"insert into runs ({','.join(row)}) values ({','.join(':'+key for key in row)})", row).lastrowid
    db.close()
    return result["run_id"]


Recording the earlier Concrete comparison into a temporary log shows the shape of a logged row. A path whose directory does not exist returns `None` and records nothing.

In [ ]:
td_db = Path(tempfile.mkdtemp())/"bench.db"
run_ids = [record_result(result, dict(lesson=True), db_path=td_db) for result in results]
logged = [dict(row) for row in bench_db(td_db).execute("select * from runs")]
test_eq([row["model"] for row in logged], [result.model for result in results])
test_eq(run_ids, [result["run_id"] for result in results])
test_eq(record_result(results[0], db_path=td_db.parent/"missing"/"bench.db"), None)
{key:logged[0][key] for key in ("ts","machine","version","dataset","model","rmse","fit")}

## Canonical dataset runs

`benchmark_dataset` applies the repository's canonical loader and split, running each model in an isolated process under one timeout. For a dataset whose canonical split is chronological, `fit_dataset` passes its `order_column` to the FastForest models automatically. `benchmark_configs` compares named FastForest parameter sets on that same path; `benchmark_datasets` collects several model comparisons, can skip completed rows, and saves after each dataset so an interrupted run remains useful. The returned `FitResults` is the report: display it directly in a notebook, or let the CLI print that same repr.

In [ ]:
def fit_dataset(
    model, # Model name
    dataset, # `Dataset` member or string id
    rf_trees=100, # sklearn RF trees
    ff_kwargs=None, # FastForest keyword overrides
    max_rows=None, # Optional reproducible dataset row limit
    data_home=".data", # Dataset cache directory
):
    "Load, canonically split, and score one model/dataset pair."
    dataset,data_home,ff_kwargs = Dataset(dataset),Path(data_home),ff_kwargs or {}
    if "order" not in ff_kwargs and (column := order_column(dataset)) is not None: ff_kwargs = {**ff_kwargs, "order": column}
    _,X,y,missing_values,task = load_data(dataset, data_home)
    y = np.asarray(y, dtype=np.float32 if task == "regression" else None)
    if max_rows is not None and max_rows < len(X):
        if max_rows < 2: raise ValueError("max_rows must be at least 2")
        selected,_ = train_test_split(np.arange(len(X)), train_size=max_rows, random_state=42,
            stratify=y if task == "classification" else None)
        X,y = _rows(X,selected),y[selected]
    train,test,split = split_indices(dataset, X, y if task == "classification" else None)
    result = fit_score(model, task, _rows(X,train), y[train], _rows(X,test), y[test], missing_values, rf_trees, ff_kwargs,
        dataset=dataset)
    result["split"],result["dataset_rows"],result["dataset_features"] = split,len(X),X.shape[1]
    return result

def benchmark_dataset(
    dataset, # `Dataset` member or string id
    models=("FastForest", "RandomForest", "HistGBM"), # Models to compare
    timeout=180, # Seconds allowed per model, including loading
    **kwargs, # Passed to `fit_dataset`
):
    "Run one dataset's canonical benchmark split."
    dataset,results = Dataset(dataset),FitResults()
    for model in models:
        result = call_timed(fit_dataset, model, dataset, timeout=timeout, **kwargs)
        result = result or FitResult(status="timeout", model=model, dataset=dataset, task=dataset_task(dataset))
        record_result(result, kwargs)
        results.append(result)
    return results

def benchmark_configs(
    dataset, # Dataset member or string id
    configs, # Mapping of result labels to FastForest keyword overrides
    base=None, # Shared FastForest keywords
    timeout=180, # Seconds allowed per configuration, including loading
    **kwargs, # Passed to fit_dataset
):
    "Compare named FastForest configurations on one canonical split."
    dataset,results = Dataset(dataset),FitResults()
    for label,overrides in configs.items():
        ff_kwargs = dict(base or {})
        ff_kwargs.update(overrides)
        result = call_timed(fit_dataset, "FastForest", dataset, timeout=timeout, ff_kwargs=ff_kwargs, **kwargs)
        if result is None: result = FitResult(status="timeout", dataset=dataset, task=dataset_task(dataset))
        result.model = label
        record_result(result, dict(ff_kwargs=ff_kwargs))
        results.append(result)
    return results


def benchmark_datasets(
    datasets, # Iterable of `Dataset` members or string ids
    models=("FastForest", "RandomForest", "HistGBM"), # Models to compare
    output=None, # Resumable full-precision CSV, saved after each dataset
    resume=False, # Skip complete dataset/model groups already in `output`
    **kwargs, # Passed to `benchmark_dataset`
):
    "Run canonical benchmarks across several datasets."
    completed = set()
    if resume and output and Path(output).exists():
        previous = pd.read_csv(output)
        completed = set(zip(previous.dataset,previous.model))
    collected = FitResults()
    for dataset in map(Dataset, datasets):
        if all((str(dataset),model) in completed for model in models): continue
        results = benchmark_dataset(dataset, models=models, **kwargs)
        collected.extend(results)
        if output: results.save_run(output)
    return collected

A canonical run loads and splits the dataset inside the isolated worker, then returns the same result objects. The checks also cover a timeout and a child error surfacing in the parent.


In [ ]:
fresh = fastforest.bench.benchmark_configs(Dataset.concrete, {"baseline":{}, "statistics":{"target_statistics":True}})
assert fresh["baseline"].dataset == Dataset.concrete
test_eq([result.model for result in fresh], ["baseline", "statistics"])
assert len(fresh["statistics"].split_frame(encoded=True)) == fresh["statistics"]["n_encoded_features"]
assert call_timed(time.sleep, 60, timeout=0.2) is None
test_fail(lambda: call_timed(int, "x"), contains="ValueError")
fresh

## Recording results

`save_results` merges a run into the benchmark CSVs that the README tables read (`tools/results/`), replacing any existing rows for that dataset and writing a timeout row where a result is `None`. `FitResults.save` is the same operation on the collection in hand, taking the task and dataset from its members. The catalog ids differ from `Dataset` names in two places: `covertype` records as `covertype_bin`, and `walmart_raw` as `walmart`. `compare(save=True)` reaches it through `promote`, publishing rows from the run log rather than from memory, and refuses non-default configurations unless forced. `AutoForest` and `Autogrow` rows are skipped when the fitted model does not differ from the base `FastForest`: no sizing was active and no tree growth was accepted.


In [ ]:
#| export
def save_results(
    results, # Mapping of model name ('FastForest', 'RandomForest', ...) to `FitResult`, or None for a timeout
    task, # 'regression' or 'classification'
    dataset, # `Dataset` member (or its string id) the results describe
    results_dir, # Directory holding regression.csv and classification.csv
):
    "Merge model results into the benchmark CSV for `task`, replacing existing rows for `dataset`."

    ids = {Dataset.covertype:"covertype_bin", Dataset.walmart_raw:"walmart"}
    dataset_id = ids.get(dataset, str(dataset))
    _check_dataset_meta(dataset_id, results, results_dir)
    path = Path(results_dir)/("classification.csv" if task == "classification" else "regression.csv")
    table = pd.read_csv(path)
    names = dict(FastForest="fastforest", AutoForest="AutoForest", Autogrow="autogrow", RandomForest="sklearn RF", HistGBM="sklearn HistGBM")
    for name,result in results.items():
        model = names[name]
        table = table[~((table.dataset == dataset_id)&(table.model == model))]
        if name in ("AutoForest","Autogrow") and result is not None and not result["sizing_active"]: continue
        row = dict(dataset=dataset_id, model=model, status="", note="")
        if result is None: row.update(status="timeout", note="timed out")
        elif task == "classification": row.update(f1=result["f1"], log_loss=result["log_loss"], fit=result["fit"], proba=result["predict"])
        else: row.update(rmse=result["rmse"], r2=result["r2"], fit=result["fit"], predict=result["predict"])
        table = pd.concat([table,pd.DataFrame([row])], ignore_index=True)
    table.to_csv(path, index=False)

`promote` publishes recorded runs: it reads them back from the run log and merges them into the CSVs through `save_results`. Promotion also checks each run's measured dataset shape against `datasets.csv`, and refuses to publish when they disagree, naming both sides. A stale metadata row then fails loudly instead of rendering a wrong README cell.

In [ ]:
#| export
def _check_dataset_meta(dataset_id, results, results_dir):
    "Refuse promotion when a run's measured dataset shape disagrees with datasets.csv."
    path = Path(results_dir)/"datasets.csv"
    if not path.exists(): return
    meta = pd.read_csv(path).set_index("dataset")
    for result in results.values():
        if result is None or "dataset_rows" not in result: continue
        if dataset_id not in meta.index: raise ValueError(f"{dataset_id} is missing from {path}")
        declared = meta.loc[dataset_id]
        features = declared.benchmark_detail.split()[0]
        if int(declared["rows"]) != result["dataset_rows"] or (features.isdigit() and int(features) != result["dataset_features"]):
            raise ValueError(f"{path} describes {dataset_id} as {int(declared['rows'])} rows, {declared.benchmark_detail}, "
                f"but the run measured {result['dataset_rows']} rows and {result['dataset_features']} features; update the metadata")

def promote(
    run_ids, # Recorded run ids to publish
    results_dir="tools/results", # README benchmark CSV directory
    db_path=BENCH_DB, # Run-log database holding the runs
):
    "Merge recorded runs into the README benchmark CSVs, one dataset at a time."
    db = bench_db(db_path)
    groups = {}
    for run_id in run_ids:
        row = db.execute("select * from runs where id=?", (run_id,)).fetchone()
        if row is None: raise ValueError(f"run {run_id} is not in {db_path}")
        groups.setdefault((row["task"], row["dataset"]), []).append(dict(row))
    db.close()
    for (task,dataset),rows in groups.items():
        results = FitResults()
        for row in rows:
            data = {key:row[key] for key in ("rmse","r2","f1","log_loss","fit","predict","n_trees","status","split",
                "dataset_rows","dataset_features") if row[key] is not None}
            data.update(json.loads(row["detail"] or "{}"))
            results.append(FitResult(data, model=row["model"], dataset=Dataset(dataset), task=task))
        results.save(results_dir)

In [ ]:
with tempfile.TemporaryDirectory() as td:
    shutil.copy("../tools/results/regression.csv", Path(td)/"regression.csv")
    shutil.copy("../tools/results/datasets.csv", Path(td)/"datasets.csv")
    promote(run_ids, td, db_path=td_db)
    merged = pd.read_csv(Path(td)/"regression.csv")
    wrong = FitResult(dict(rmse=5.8, fit=.1, predict=.01, dataset_rows=99, dataset_features=8),
        model="FastForest", dataset=Dataset.concrete, task="regression")
    test_fail(lambda: FitResults([wrong]).save(td), contains="update the metadata")
concrete_rows = merged[merged.dataset == "concrete"]
test_eq(set(concrete_rows.model), {"fastforest", "sklearn RF", "sklearn HistGBM"})
concrete_rows[["dataset", "model", "rmse", "r2", "fit", "predict"]]


,dataset,model,rmse,r2,fit,predict
39,concrete,fastforest,5.697535,0.874023,0.004437,0.000533
40,concrete,sklearn RF,5.473875,0.883719,0.050747,0.012602
41,concrete,sklearn HistGBM,4.645888,0.916236,1.179723,0.006728


## The full comparison

`compare` is the complete harness behind the README tables: it selects datasets and models, applies FastForest overrides, and can merge the results into the benchmark CSVs. `tools/accuracy.py`, a thin `call_parse` wrapper around it, gives the CLI and an interactive kernel one shared implementation and one set of docments.


In [ ]:
#| export
README_DATASETS = tuple(map(Dataset, (
    "sgemm", "california", "concrete", "diamonds", "allstate", "diabetes", "bluebook", "walmart", "rossmann", "ashrae",
    "covertype", "adult", "bank", "click", "shuttle", "airlines", "higgs", "kddcup99", "sf_police")))

def _max_features(value):
    "Parse square-root or fractional feature selection."
    if value == "sqrt": return value
    return float(value)

def _replacement(value):
    value = value.lower()
    if value == "none": return None
    if value == "true": return True
    if value == "false": return False
    raise ValueError("replacement must be none, true, or false")

def compare(
    dataset:Dataset=Dataset.california, # Regression or classification dataset
    datasets:str=None,             # Comma-separated datasets, or readme for all displayed datasets
    rf_trees:int=100,              # sklearn RF trees (its default)
    ff_trees:int=None,             # FastForest trees; defaults to its sampled-row rule
    min_node_size:int=None,        # FastForest minimum node size; model default when omitted
    bootstrap_fraction:float=None, # Defaults to 1, or 0.8 with OOB
    bootstrap_max:int=None,        # Maximum sampled rows per output; model default when omitted
    replacement:str="none",       # none is adaptive; true or false overrides it
    max_node_samples:int=None,     # Maximum rows evaluated per node; model default when omitted
    split_prior_rows:float=None,   # Prior rows used by the split score; model default when omitted
    class_weight_power:float=None, # Classification inverse-frequency weighting; model default when omitted
    cutoff_divisor:float=None,     # No-sort splitter candidate-count divisor; model default when omitted
    random_splitter:bool=False,    # Use the original random split search
    max_features:str=None,         # sqrt or a feature fraction; task default when omitted
    dates:bool=True,               # Let fastforest auto-detect dates
    target_statistics:bool=True,  # Add per-level target statistics where eligible
    min_rows_per_level:int=None,   # Minimum training rows supporting a derived level
    min_stat_agreement:float=None, # Minimum deployment-split agreement of statistic ranks
    frequency:bool=False,          # Add exact per-level frequency features
    natural_sort:bool=True,        # Sort text digit runs numerically
    timeout:int=180,               # Maximum seconds for each model/dataset combination, including loading
    models:str=None,               # Comma-separated model names; default FastForest,RandomForest,HistGBM
    max_rows:int=None,             # Optional reproducible dataset row limit
    data_home:str=".data",         # Dataset cache directory
    save:bool=False,               # Update the README benchmark result CSVs
    results_dir:str="tools/results", # README benchmark CSV directory written by `save`
    output:str=None,               # Resumable CSV collecting this run's full-precision results
    resume:bool=False,             # Skip dataset/model combinations already present in output
    force:bool=False,              # Allow save with a non-default model configuration
):
    "Compare accuracy and timing on canonical dataset splits."
    if timeout < 1: raise ValueError("timeout must be positive")
    models = models.split(",") if models else ["FastForest", "RandomForest", "HistGBM"]
    unknown = set(models) - {"FastForest", "RandomForest", "HistGBM", "AutoForest", "Autogrow"}
    if unknown: raise ValueError(f"unknown models {sorted(unknown)}")
    ff_kwargs = dict(n_trees=ff_trees, bootstrap_fraction=bootstrap_fraction, replacement=_replacement(replacement),
        random_splitter=random_splitter, date_columns=None if dates else {}, target_statistics=target_statistics,
        frequency=frequency, natural_sort=natural_sort)
    if max_features is not None: ff_kwargs["max_features"] = _max_features(max_features)
    for name,value in dict(min_node_size=min_node_size, bootstrap_max=bootstrap_max, max_node_samples=max_node_samples,
        split_prior_rows=split_prior_rows, class_weight_power=class_weight_power, cutoff_divisor=cutoff_divisor,
        min_rows_per_level=min_rows_per_level, min_stat_agreement=min_stat_agreement,
        ).items():
        if value is not None: ff_kwargs[name] = value
    canonical = (len(ff_kwargs) == 8 and ff_kwargs["n_trees"] is None and ff_kwargs["bootstrap_fraction"] is None
        and ff_kwargs["replacement"] is None and not ff_kwargs["random_splitter"] and ff_kwargs["date_columns"] is None
        and ff_kwargs["target_statistics"] and not ff_kwargs["frequency"] and ff_kwargs["natural_sort"])
    if save and not force and not (canonical and max_rows is None and rf_trees == 100):
        raise ValueError("save promotes README rows and requires the default configuration; pass force to promote anyway")
    selected = README_DATASETS if datasets == "readme" else tuple(map(Dataset, datasets.split(","))) if datasets else (dataset,)
    results = benchmark_datasets(selected, models=models, output=output, resume=resume, timeout=timeout, rf_trees=rf_trees,
        ff_kwargs=ff_kwargs, max_rows=max_rows, data_home=data_home)
    if save:
        run_ids = [result["run_id"] for result in results if "run_id" in result]
        if not run_ids: raise RuntimeError("no recorded runs to promote; run from the repository root")
        promote(run_ids, results_dir)
    if is_cli(): print(results)
    else: return results

For large datasets, `python tools/accuracy.py --dataset <name>` runs this same machinery with each model in an isolated process under a 180-second timeout, and writes the CSVs itself.